In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
  !ls -R /kaggle/input | head -30


In [2]:
!pip install unsloth
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas
!pip install --upgrade --force-reinstall Pillow

  Using cached unsloth-2026.11.1-py3-none-any.whl.metadata (80 kB)
  Using cached unsloth_zoo-2026.11.1-py3-none-any.whl.metadata (33 kB)
  Using cached tyro-1.0.16-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 103.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 146.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 86.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [3]:
import os, re, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from unsloth import FastVisionModel

ADAPTER = "/kaggle/input/datasets/varadgaikwad17/receipt-model"   # <- the folder that holds adapter_config.json
model, processor = FastVisionModel.from_pretrained(ADAPTER, load_in_4bit=True)
processor.image_processor.max_soft_tokens = 560
processor.image_processor.image_seq_length = 560
FastVisionModel.for_inference(model)
print("loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.1: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

loaded


In [8]:
INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."

def shrink(img):
    img = img.convert("RGB"); img.thumbnail((1792, 1792)); return img

def parse_json(text):
    m = re.search(r"\{.*?\}", text, re.DOTALL)
    try: return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError: return None

def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=200):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    txt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inp = processor(image.convert("RGB"), txt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [9]:
import base64
import os
import gradio as gr

gr.close_all()

# Load cooked_logo.png as inline base64 if present in directory
logo_src = ""
if os.path.exists("cooked_logo.png"):
  with open("cooked_logo.png", "rb") as f:
    logo_src = f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# Comprehensive CSS: vector entrance, pulsing dots, animated line bridges, and zero white boxes
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700;800;900&family=JetBrains+Mono:wght@500;700&display=swap');

html { scroll-behavior: smooth; }
body, .gradio-container {
    background-color: #0b0f17 !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    color: #f1f5f9 !important;
}

/* FIX CODEMIRROR & GRADIO INNER WHITE BOXES */
.cm-editor, .cm-scroller, .cm-gutters, .cm-content, .cm-line {
    background-color: #05080f !important;
    color: #38bdf8 !important;
    border-color: #1e293b !important;
}
.cm-gutters {
    border-right: 1px solid #1e293b !important;
    color: #475569 !important;
}
.gradio-container textarea, .gradio-container .cm-editor {
    border: 1px solid #1e293b !important;
    border-radius: 14px !important;
    background: #05080f !important;
}

/* REMOVE WHITE BOXES INSIDE GRADIO UPLOAD CONTAINER */
.gradio-container .image-container,
.gradio-container .upload-container,
.gradio-container .empty,
.gradio-container [data-testid="image"],
.gradio-container .wrap {
    background-color: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

/* ANIMATIONS */
@keyframes vector-enter-left {
  0% { opacity: 0; transform: translateX(-40px) scale(0.92); }
  100% { opacity: 1; transform: translateX(0) scale(1); }
}
@keyframes vector-enter-right {
  0% { opacity: 0; transform: translateX(40px) scale(0.92); }
  100% { opacity: 1; transform: translateX(0) scale(1); }
}
@keyframes line-draw {
  0% { stroke-dashoffset: 100; opacity: 0.2; }
  50% { opacity: 1; }
  100% { stroke-dashoffset: 0; opacity: 1; }
}
@keyframes pulse-ring {
  0% { transform: scale(0.97); box-shadow: 0 0 0 0 rgba(249, 115, 22, 0.45); }
  70% { transform: scale(1); box-shadow: 0 0 0 16px rgba(249, 115, 22, 0); }
  100% { transform: scale(0.97); box-shadow: 0 0 0 0 rgba(249, 115, 22, 0); }
}
@keyframes pulse-dot {
  0%, 100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); }
  50% { transform: scale(1.35); box-shadow: 0 0 10px 3px rgba(16, 185, 129, 0.9); }
}
@keyframes pulse-dot-plain {
  0%, 100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 255, 255, 0.5); }
  50% { transform: scale(1.35); box-shadow: 0 0 8px 3px rgba(255, 255, 255, 0.8); }
}

.anim-vector-left { animation: vector-enter-left 1.2s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
.anim-vector-right { animation: vector-enter-right 1.2s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
.dot-tuned { animation: pulse-dot 2s infinite ease-in-out; }
.dot-plain { animation: pulse-dot-plain 2s infinite ease-in-out; }
.connecting-line { stroke-dasharray: 100; animation: line-draw 2s ease-in-out infinite alternate; }

/* Sticky Navbar */
.cooked-nav {
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: rgba(11, 15, 23, 0.85);
    backdrop-filter: blur(12px);
    border: 1px solid #1e293b;
    border-radius: 20px;
    padding: 12px 28px;
    margin-bottom: 25px;
    position: sticky;
    top: 10px;
    z-index: 100;
}
.nav-links a {
    color: #94a3b8;
    text-decoration: none;
    font-weight: 700;
    font-size: 14px;
    margin-left: 20px;
    transition: color 0.2s;
}
.nav-links a:hover { color: #f97316; }

/* Circular Upload Dropzone */
.circular-upload {
    border: 2px dashed #f97316 !important;
    background: radial-gradient(circle, #172233 0%, #0d1421 100%) !important;
    border-radius: 28px !important;
    padding: 24px !important;
    animation: pulse-ring 3.5s infinite;
}

/* Action Button */
.extract-btn {
    background: linear-gradient(90deg, #f97316, #ea580c) !important;
    color: #0b0f17 !important;
    font-weight: 900 !important;
    font-size: 16px !important;
    border-radius: 14px !important;
    border: none !important;
    padding: 16px !important;
    box-shadow: 0 10px 25px -5px rgba(249, 115, 22, 0.4) !important;
}
"""

def pretty(raw):
    parsed = parse_json(raw)
    return json.dumps(parsed, indent=2, ensure_ascii=False) if parsed else raw
def generate_plain_vision(image):
    try:
        with model.disable_adapter():
            return pretty(generate_from_image(shrink(image)))
    except Exception as e:
        return f"(Plain model output unavailable: {e})"


def on_compare_image(image):
    if image is None:
        raise gr.Error("Please upload or choose a receipt image first.")
    return generate_plain_vision(image), pretty(generate_from_image(shrink(image)))


with gr.Blocks(
    title="cooked.exe - Raw receipts in. Structured JSON out.", css=CUSTOM_CSS
) as demo:

  # 1. FIXED TOP NAVBAR
  logo_html = (
      f'<img src="{logo_src}" style="height:36px; object-fit:contain;" />'
      if logo_src
      else '<span style="font-family:monospace; font-size:22px; font-weight:900; color:white;">cooked<span style="color:#f97316;">.exe</span></span>'
  )
  gr.HTML(f"""
        <div class="cooked-nav">
            <div style="display:flex; align-items:center; gap:12px;">
                {logo_html}
            </div>
            <div class="nav-links">
                <a href="#demo" style="color:#f97316;">Home</a>
                <a href="#demo">Live Extractor</a>
                <a href="#benchmarks">Benchmark</a>
                <a href="#architecture">Architecture</a>
            </div>
        </div>
    """)

  # 2. HERO HEADLINE
  gr.HTML("""
        <div style="text-align:center; padding: 15px 0 25px 0;">
            <div style="display:inline-block; padding: 4px 14px; border-radius:999px; background:rgba(249,115,22,0.12); border:1px solid rgba(249,115,22,0.3); color:#f97316; font-size:12px; font-weight:bold; font-family:'JetBrains Mono'; margin-bottom:12px;">
                GEMMA 4 VISION E2B LoRA
            </div>
            <h1 style="font-size: 3rem; font-weight: 900; line-height: 1.15; margin: 0 0 10px 0;">
                Raw receipts in.<br/>
                <span style="background: linear-gradient(90deg, #fb923c, #f59e0b); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
                    Structured JSON out.
                </span>
            </h1>
            <p style="color: #94a3b8; font-size: 15px; max-width: 520px; margin: 0 auto;">
                Eliminate manual data entry. Extract company, date, address, and total amount in seconds.
            </p>
        </div>
    """)

  # 3. HERO INTERACTIVE ARENA: VECTORS + CIRCULAR UPLOAD
  with gr.Row(elem_id="demo"):
    # Left Vector (Entrance animation on page load)
    with gr.Column(scale=1, min_width=220):
      gr.HTML("""
                <div class="anim-vector-left" style="display:flex; flex-direction:column; align-items:center; justify-content:center; height:100%;">
                    <svg style="width:200px; height:200px;" viewBox="0 0 240 240" fill="none">
                        <line x1="20" y1="40" x2="220" y2="40" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <line x1="20" y1="120" x2="220" y2="120" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <line x1="20" y1="200" x2="220" y2="200" stroke="#1e293b" stroke-width="1" stroke-dasharray="4 4"/>
                        <polygon points="50,60 190,60 170,180 30,180" fill="rgba(30, 41, 59, 0.4)" stroke="#334155" stroke-width="1.5"/>
                        <polygon points="65,75 175,75 160,165 50,165" fill="rgba(15, 23, 42, 0.8)" stroke="#475569" stroke-width="1.5"/>
                        <line x1="75" y1="95" x2="145" y2="95" stroke="#f97316" stroke-width="3" stroke-linecap="round"/>
                        <line x1="73" y1="110" x2="125" y2="110" stroke="#64748b" stroke-width="2" stroke-linecap="round"/>
                        <line x1="71" y1="125" x2="140" y2="125" stroke="#64748b" stroke-width="2" stroke-linecap="round"/>
                        <line x1="69" y1="140" x2="110" y2="140" stroke="#10b981" stroke-width="2.5" stroke-linecap="round"/>
                        <line x1="25" y1="105" x2="205" y2="105" stroke="#f97316" stroke-width="2"/>
                        <circle cx="120" cy="120" r="70" stroke="rgba(249, 115, 22, 0.25)" stroke-width="1.5" stroke-dasharray="8 8"/>
                    </svg>
                    <div style="font-family:'JetBrains Mono'; font-size:11px; color:#94a3b8; margin-top:8px;">
                        <span style="color:#f97316; font-weight:bold;">566</span> Trained Receipts
                    </div>
                </div>
            """)

    # Center Big Upload
    with gr.Column(scale=3):
      image_input = gr.Image(
          type="pil",
          label="Drop image here or click to upload",
          show_label=False,
          sources=["upload", "clipboard"],
          elem_classes=["circular-upload"],
      )

      gr.HTML("""
                <div style="text-align:center; margin-top:8px; margin-bottom:12px; font-size:12px; color:#94a3b8;">
                    Drag or upload a receipt image (PNG or JPG)
                </div>
            """)

      extract_btn = gr.Button(
          "Run Side-by-Side Extraction",
          variant="primary",
          elem_classes=["extract-btn"],
      )

    # Right Vector (Entrance animation on page load)
    with gr.Column(scale=1, min_width=220):
      gr.HTML("""
                <div class="anim-vector-right" style="display:flex; flex-direction:column; align-items:center; justify-content:center; height:100%;">
                    <svg style="width:200px; height:200px;" viewBox="0 0 240 240" fill="none">
                        <polygon points="120,40 180,75 180,145 120,180 60,145 60,75" stroke="#334155" stroke-width="1.5" fill="rgba(15, 23, 42, 0.6)"/>
                        <polygon points="120,60 160,85 160,135 120,160 80,135 80,85" stroke="#10b981" stroke-width="1.5" stroke-dasharray="4 4" fill="rgba(16, 185, 129, 0.05)"/>
                        <circle cx="120" cy="40" r="4" fill="#f97316"/>
                        <circle cx="180" cy="75" r="4" fill="#38bdf8"/>
                        <circle cx="180" cy="145" r="4" fill="#10b981"/>
                        <circle cx="120" cy="180" r="4" fill="#f97316"/>
                        <circle cx="60" cy="145" r="4" fill="#38bdf8"/>
                        <circle cx="60" cy="75" r="4" fill="#10b981"/>
                        <line x1="120" y1="40" x2="120" y2="180" stroke="rgba(249, 115, 22, 0.3)" stroke-width="1"/>
                        <line x1="60" y1="75" x2="180" y2="145" stroke="rgba(56, 189, 248, 0.3)" stroke-width="1"/>
                        <line x1="180" y1="75" x2="60" y2="145" stroke="rgba(16, 185, 129, 0.3)" stroke-width="1"/>
                    </svg>
                    <div style="font-family:'JetBrains Mono'; font-size:11px; color:#94a3b8; margin-top:8px;">
                        LoRA Rank <span style="color:#c084fc; font-weight:bold;">16</span> • <span style="color:#34d399; font-weight:bold;">29.9M</span> Params
                    </div>
                </div>
            """)

  # 4. SIDE-BY-SIDE MODEL OUTPUTS (FIXED DARK THEME)
  with gr.Row():
    plain_out = gr.Code(
        value=(
            "// Plain Gemma 4 Vision output will appear here after"
            " extraction..."
        ),
        label="Plain Gemma 4 Vision (Baseline)",
        language="json",
        lines=8,
    )
    tuned_out = gr.Code(
        value=(
            "// cooked.exe Fine-Tuned Gemma output will appear here after"
            " extraction..."
        ),
        label="cooked.exe Fine-Tuned Gemma (LoRA)",
        language="json",
        lines=8,
    )

  extract_btn.click(
      on_compare_image, inputs=image_input, outputs=[plain_out, tuned_out]
  )

  # 5. FIXED BENCHMARKS: CONNECTED HISTOGRAM WITH ANIMATED DOT-BRIDGES
  gr.HTML("""
        <div id="benchmarks" style="padding-top:60px; margin-top:40px; border-top:1px solid #1e293b;">
            <div style="display:flex; justify-content:space-between; align-items:flex-end; margin-bottom:16px;">
                <div>
                    <h2 style="font-size:26px; font-weight:900; color:white; margin:0 0 4px 0;">Accuracy on 361 Test Receipts (Normalized Match)</h2>
                    <p style="color:#94a3b8; font-size:13px; margin:0;">Direct evaluation on unseen Malaysian retail receipts (jsdnrs/ICDAR2019-SROIE).</p>
                </div>
                <div style="display:flex; gap:16px; font-size:12px;">
                    <div style="display:flex; align-items:center; gap:6px;">
                        <span style="width:12px; height:12px; background:#64748b; border-radius:3px;"></span>
                        <span style="color:white; font-weight:bold;">Plain</span>
                    </div>
                    <div style="display:flex; align-items:center; gap:6px;">
                        <span style="width:12px; height:12px; background:#10b981; border-radius:3px;"></span>
                        <span style="color:#34d399; font-weight:bold;">Fine-Tuned</span>
                    </div>
                </div>
            </div>

            <!-- HISTOGRAM CONTAINER WITH CONNECTED DOT BRIDGES -->
            <div style="background:#0e1524; border:1px solid #22324f; border-radius:18px; padding:45px 20px 20px 20px; box-shadow:0 15px 35px rgba(0,0,0,0.4);">

                <div style="display:flex; justify-content:space-around; align-items:flex-end; height:240px; border-bottom:1px solid #334155; position:relative;">

                    <!-- GROUP 1: COMPANY NAME -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-40px; left:0; right:0; width:100%; height:50px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 38 C 38 32 52 13 69 6" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (74.2%) -->
                            <div style="width:46px; height:178px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">74.2%</span>
                            </div>
                            <!-- Fine-Tuned Bar (87.5%) -->
                            <div style="width:46px; height:210px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">87.5%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Company Name</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+13.3 points</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 2: DATE EXTRACTION -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-80px; left:0; right:0; width:100%; height:90px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 78 C 38 66 52 21 69 6" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (67.3%) -->
                            <div style="width:46px; height:162px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">67.3%</span>
                            </div>
                            <!-- Fine-Tuned Bar (97.5%) -->
                            <div style="width:46px; height:234px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">97.5%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Date Extraction</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+30.2 points</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 3: ADDRESS DETECTION (STEEP 2X RISE LINE) -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 (Dramatic rise) -->
                            <svg style="position:absolute; top:-30px; left:0; right:0; width:100%; height:40px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 28 C 38 23 52 11 69 6" fill="none" stroke="#10b981" stroke-width="3" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (70.9%) -->
                            <div style="width:46px; height:170px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">70.9%</span>
                            </div>
                            <!-- Fine-Tuned Bar (79.8%) -->
                            <div style="width:46px; height:192px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">79.8%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Address Detection</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399; font-weight:bold;">+8.9 points</span>
                    </div>

                    <!-- SPACE -->

                    <!-- GROUP 4: TOTAL AMOUNT -->
                    <div style="display:flex; flex-direction:column; align-items:center;">
                        <div style="display:flex; align-items:flex-end; position:relative;">

                            <!-- Animated line bridging dot 1 and dot 2 -->
                            <svg style="position:absolute; top:-47px; left:0; right:0; width:100%; height:57px; pointer-events:none; overflow:visible; z-index:20;">
                                <path d="M 23 45 C 38 38 52 15 69 6" fill="none" stroke="#10b981" stroke-width="2.5" stroke-linecap="round" class="connecting-line"/>
                            </svg>

                            <!-- Plain Bar (78.1%) -->
                            <div style="width:46px; height:187px; background:linear-gradient(to top, rgba(51,65,85,0.4), #475569); border-top-left-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-plain" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:white; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:bold; color:white; margin-top:6px;">78.1%</span>
                            </div>
                            <!-- Fine-Tuned Bar (94.2%) -->
                            <div style="width:46px; height:226px; background:linear-gradient(to top, rgba(16,185,129,0.3), #10b981); border-top-right-radius:6px; position:relative; display:flex; justify-content:center;">
                                <div class="dot-tuned" style="position:absolute; top:-5px; width:10px; height:10px; border-radius:50%; background:#34d399; border:2px solid #090d16; z-index:30;"></div>
                                <span style="font-size:11px; font-family:monospace; font-weight:900; color:#ecfdf5; margin-top:6px;">94.2%</span>
                            </div>
                        </div>
                        <span style="font-size:12px; font-weight:bold; color:#e2e8f0; margin-top:12px;">Total Amount</span>
                        <span style="font-size:10px; font-family:monospace; color:#34d399;">+16.1 points</span>
                    </div>

                </div>

                <!-- TABLE: EXACT ONLY, PLAIN TEXT IN CLEAN WHITE -->
                <div style="margin-top:24px; overflow-x:auto;">
                    <table style="width:100%; font-size:11px; text-align:left; border-collapse:collapse; font-family:'JetBrains Mono';">
                        <thead>
                            <tr style="color:#94a3b8; border-bottom:1px solid #1e293b;">
                                <th style="padding:10px;">Field</th>
                                <th style="padding:10px; color:white; font-weight:bold;">Plain</th>
                                <th style="padding:10px; color:#34d399; font-weight:bold;">Fine-Tuned</th>
                                <th style="padding:10px; color:#34d399; font-weight:bold;">Gain (points)</th>
                            </tr>
                        </thead>
                        <tbody style="color:#cbd5e1;">
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-weight:bold; color:#e2e8f0;">company</td>
                                <td style="padding:10px; color:white;">74.2%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">87.5%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+13.3</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b; background:rgba(16,185,129,0.08);">
                                <td style="padding:10px; font-weight:bold; color:#e2e8f0;">date</td>
                                <td style="padding:10px; color:white;">67.3%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">97.5%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+30.2</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-weight:bold; color:#e2e8f0;">address</td>
                                <td style="padding:10px; color:white;">70.9%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">79.8%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+8.9</td>
                            </tr>
                            <tr style="border-bottom:1px solid #1e293b;">
                                <td style="padding:10px; font-weight:bold; color:#e2e8f0;">total</td>
                                <td style="padding:10px; color:white;">78.1%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">94.2%</td>
                                <td style="padding:10px; color:#34d399; font-weight:bold;">+16.1</td>
                            </tr>
                            <tr>
                                <td style="padding:10px; font-weight:bold; color:white;">average</td>
                                <td style="padding:10px; color:white; font-weight:bold;">72.6%</td>
                                <td style="padding:10px; color:#34d399; font-weight:900;">89.8%</td>
                                <td style="padding:10px; color:#34d399; font-weight:900;">+17.2</td>
                            </tr>
                        </tbody>
                    </table>
                    <p style="color:#94a3b8; font-size:11px; line-height:1.6; margin:16px 0 0 0; font-family:sans-serif;">
                        Normalized match: case, punctuation and spacing are ignored, totals are compared as numbers, and a company name at the start of the address is removed.
                        Strict exact match for the fine-tuned model: 84.8% average (company 86.1, date 97.5, address 65.1, total 90.3).
                        A detailed prompt on the plain model did not close the gap: on the first 100 test receipts it scored 70.5% on average, versus 76.0% for the plain model and 88.8% for the fine-tuned model.
                    </p>
                </div>

            </div>
        </div>
    """)

  # 6. SETUP & ARCHITECTURE SECTION
  gr.HTML("""
        <div id="architecture" style="padding-top:60px; margin-top:40px; border-top:1px solid #1e293b; padding-bottom:50px;">
            <h2 style="font-size:26px; font-weight:900; color:white; margin:0 0 4px 0;">Model Setup & Architecture</h2>
            <p style="color:#94a3b8; font-size:13px; margin:0 0 16px 0;">Hyperparameters from the fine-tuning run on a single NVIDIA T4.</p>

            <div style="display:grid; grid-template-columns: repeat(2, 1fr); gap:12px; font-family:'JetBrains Mono'; font-size:12px;">
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Base Model</span>
                    <span style="color:white; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">unsloth/gemma-4-E2B-it (4-bit)</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">LoRA Parameters</span>
                    <span style="color:#c084fc; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">r=16, 29.9M params (0.58%)</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Training Run</span>
                    <span style="color:#fbbf24; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">1 epoch, effective batch 4, 560 image tokens</span>
                </div>
                <div style="background:#131c2e; border:1px solid #22324f; border-radius:12px; padding:14px; display:flex; justify-content:space-between; align-items:center;">
                    <span style="color:#94a3b8; font-family:sans-serif;">Dataset</span>
                    <span style="color:#34d399; font-weight:bold; background:#0b0f17; padding:4px 8px; border-radius:6px;">ICDAR2019-SROIE (566 train / 361 test)</span>
                </div>
            </div>
        </div>
    """)

demo.launch(share=True)

/tmp/ipykernel_253/1149837511.py:149: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://358aca11fcee524683.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
